# YOLOv12 객체 탐지 학습

- 설명: YOLOv12s 모델을 학습하고 검증하는 초기 안전고리 객체 탐지 실험입니다.
- 작성자: 조은나라

> 공개 저장소에는 데이터, 모델 가중치, API 키와 노트북 실행 결과를 포함하지 않습니다.


In [ ]:
!pip install roboflow

import os
from roboflow import Roboflow
ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY")
if not ROBOFLOW_API_KEY:
    raise RuntimeError("ROBOFLOW_API_KEY 환경변수가 설정되지 않았습니다.")
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("korean-bus").project("test")
dataset = project.version(1).download("yolov12")

In [ ]:
!gdown 1l9SHPyK5RFdUuYviW-k-YbXYN-PhIkI-

In [ ]:
# 압축 파일을 풀면 됨!!!
# ===> unzip, -qq, -d
!unzip -qq '/content/hook_merged.zip' -d '/content/'

In [ ]:
!pip install -U ultralytics

In [ ]:
import os

print(os.listdir("/content/hook_merged"))
print(os.path.exists("/content/hook_merged/data.yaml"))

In [ ]:
!find /content/hook_merged -name "data.yaml"

In [ ]:
import yaml
from pathlib import Path

yaml_path = Path("/content/hook_merged/data.yaml")

# 기존 yaml 읽기
with open(yaml_path, "r", encoding="utf-8") as f:
    data = yaml.safe_load(f)

print("수정 전:")
print(data)

# Colab 경로로 수정
data["path"] = "/content/hook_merged"
data["train"] = "train/images"
data["val"] = "valid/images"
data["test"] = "test/images"

# 다시 저장
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(data, f, allow_unicode=True, sort_keys=False)

print("\n수정 후:")
print(data)

In [ ]:
!cat /content/hook_merged/data.yaml

In [ ]:
from ultralytics import YOLO

# YOLO12s 사전학습 모델 불러오기
model = YOLO("yolo12s.pt")

# 학습
results = model.train(
    data="/content/hook_merged/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,      # GPU 사용
    workers=2,
    project="/content/runs",
    name="hook_yolo12s"
)

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/hook_yolo12s/weights/best.pt")

results = model.predict(
    source="/content/hook_merged/test/images",  # 테스트 이미지 폴더
    imgsz=640,
    conf=0.25,
    save=True
)

In [ ]:
!ls /content/runs/detect/predict

In [ ]:
from IPython.display import Image, display
import glob

result_images = glob.glob("/content/runs/detect/predict/*.jpg")

for img_path in result_images[:10]:
    print(img_path)
    display(Image(filename=img_path))

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/hook_yolo12s/weights/best.pt")

results = model.predict(
    source=uploaded,
    imgsz=640,
    conf=0.25,
    save=True,
    project="/content/runs/detect",
    name="new_image_test"
)

In [ ]:
!find /content/runs -name "best.pt"

In [ ]:
import os

uploaded_img = list(uploaded.keys())[0]
uploaded_img_path = "/content/" + uploaded_img

print("업로드 이미지 경로:", uploaded_img_path)
print("파일 존재:", os.path.exists(uploaded_img_path))

In [ ]:
from IPython.display import Image, display
import glob

result_images = glob.glob("/content/runs/detect/new_image_test/*.jpg")

print("결과 이미지 개수:", len(result_images))
print(result_images)

display(Image(filename=result_images[0]))

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/hook_yolo12s/weights/best.pt")

results = model.predict(
    source="/content/안전고리.jpg",
    imgsz=640,
    conf=0.25,
    save=True,
    project="/content/runs/detect",
    name="new_image_test2"
)

for r in results:
    print("탐지 개수:", len(r.boxes))
    print(r.boxes)